<a href="https://colab.research.google.com/github/Madisxnki/project2-tcga-p53-capstone/blob/main/03_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Notebook 03 — Extension

**Worth 25 marks. Choose ONE.**

---

> **Read this first.** The extension is not required to do well. A
> complete, careful core — Notebooks 01 and 02, properly written and
> properly interpreted — is first-class work and scores around 70. This
> notebook is how you move from 70 to 85.
>
> **One extension. Not two, not all three.** Doing one well beats doing
> three badly, and there are no marks for volume.

All three use the same expression file as Notebook 02, and **none of
them need your earlier answers to be correct** — a working
`compare_groups` is given to you below.

## Every option has the same basic deliverables

Whichever option you choose, you must provide:

1. the named function, following the contract given;
2. the supplied structural checks passing;
3. **at least two useful checks of your own** — choose properties that
   would catch a plausible mistake, not merely `result is not None`;
4. a short interpretation explaining what the extension changes about
   your Notebook 02 conclusion.

The supplied checks are only a starting point. Choosing what else needs
testing is part of the extension.

> ### ⚠️ Run the setup, then only your chosen option
>
> Run the setup cells at the top first. After that, work through **Option
> A, B or C** — not all three. The untouched options still contain
> `NotImplementedError`, so **Restart and run all is not appropriate for
> this notebook** until you remove or skip the two options you did not
> choose.
>
> If you edit a function, re-run its cell before running its checks.

## Run this first

> ✏️ **Before you run the cell below, add your Sussex username.**
>
> Scroll to the **bottom** of the cell — **line 86 of 93** —
> and type your Sussex username (the start of your Sussex email, the
> one you log in with) between the quotes:
>
> ```python
> STUDENT_ID = "ab123"   # <-- your Sussex username
> ```
>
> Then run the cell (**Shift + Enter**). It prints your gene and cancer
> type. Change nothing else in that cell.
>
> 💡 To see line numbers in Colab: **Tools → Settings → Editor → Show
> line numbers**.

In [6]:
# Run this first. It downloads your data and sets up the self-checks.
import hashlib
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/tcga-capstone-2026/main/data/"

def fetch(filename):
    """Return a local data path, downloading it in Colab if needed."""
    repo_path = Path("data") / filename
    working_path = Path(filename)

    if repo_path.exists():                   # running from the repo
        return repo_path
    if not working_path.exists():            # running in Colab
        urllib.request.urlretrieve(DATA_URL + filename, working_path)
    return working_path


def my_variant(student_id):
    """Look up your gene and cancer type from your Sussex username."""
    number = str(student_id).strip().lower()
    if not number:
        raise ValueError("Type your Sussex username into "
                         "STUDENT_ID, then run this cell again.")
    key = hashlib.pbkdf2_hmac("sha256", number.encode(), b'tcga-capstone-2026',
                              2_000_000).hex()[:16]
    lookup = json.loads(Path(fetch("variant_lookup.json")).read_text())
    if key not in lookup:
        raise ValueError(f"{number} is not on the class list. Check it "
                         "against Canvas; if it is right, email the convenor.")
    gene, cancer_type = lookup[key]
    print(f"{number}: your gene is {gene}, your cancer type is {cancer_type}")
    return gene, cancer_type


def run_checks(checks):
    """Run a dict of {description: test} and report what passed.

    Each test is a zero-argument function returning True or False. A
    test that raises is reported rather than stopping the notebook, so
    you always see the full picture.
    """
    passed = 0
    for description, test in checks.items():
        try:
            ok = bool(test())
        except Exception as exc:
            print(f"\u274c {description}")
            print(f"     raised {type(exc).__name__}: {exc}")
            continue
        print(f"{'\u2705' if ok else '\u274c'} {description}")
        passed += ok
    print()
    print(f"{passed} of {len(checks)} checks passed")
    if passed < len(checks):
        print("Keep going - fix the crosses and run this cell again.")


# GIVEN - a working compare_groups, so the extension does not depend on
# whether yours came out right in Notebook 02. Use it.
def compare_groups(expr, group_col, group_a, group_b):
    """Compare expression between two groups with a Mann-Whitney U test."""
    a = expr.loc[expr[group_col] == group_a, "log2_tpm"].dropna()
    b = expr.loc[expr[group_col] == group_b, "log2_tpm"].dropna()
    if a.empty or b.empty:
        raise ValueError(
            f"empty group: {group_a!r} has {len(a)} rows, "
            f"{group_b!r} has {len(b)}"
        )
    p_value = float(mannwhitneyu(a, b, alternative="two-sided").pvalue)
    return {
        "n_a": int(len(a)), "n_b": int(len(b)),
        "median_a": float(a.median()), "median_b": float(b.median()),
        "difference": float(a.median() - b.median()),
        "p_value": p_value, "significant": bool(p_value < 0.05),
    }


STUDENT_ID = "wk202"   # <-- your Sussex username, e.g. "ab123"

GENE, CANCER_TYPE = my_variant(STUDENT_ID)

expr = pd.read_csv(fetch(f"expression_{GENE}_{CANCER_TYPE}.csv"))
tumours = expr[(expr["cancer_type"] == CANCER_TYPE)
               & (expr["group"] == "tumour")].copy()
print(len(tumours), "tumours in", CANCER_TYPE)

wk202: your gene is CCNB1, your cancer type is LUAD
513 tumours in LUAD


---

# Option C — Does it hold in a different tissue?

**The most straightforward, and available to everybody.**

You tested one cancer type. Is what you found a general property of p53
biology, or something specific to that tissue?

The expression file contains tumours from **twelve** cancer types.
Repeat your TP53 comparison in each and see where your gene behaves the
same way and where it does not.

Two things to be careful about, both marked:

- **Group sizes vary wildly.** TP53 is mutated in most oesophageal
  tumours and very few prostate ones. A comparison with eight mutant
  tumours is not evidence. Set a minimum and say what it is.
- **Twelve tests means twelve chances of a false positive.** At
  p < 0.05, you expect roughly one significant result by luck alone. Say
  something sensible about that — you need not apply a formal
  correction, but you do have to show you know the problem exists.

## The contract

```python
compare_across_cancers(expr, min_group_size)
```

**Takes** `expr`, the tidy expression table, and the minimum number of
mutant **and** wild-type tumours required for a cancer type to be kept.
Reuse the given `compare_groups` function for each eligible cancer.

**Returns** a DataFrame with exactly these columns:

```text
cancer_type, n_mut, n_wt, difference, p_value, significant
```

There should be one row per eligible cancer type, sorted by `p_value`
smallest first, with the index reset. `difference` is mutant minus
wild-type expression.

In [7]:
def compare_across_cancers(expr, min_group_size):
    """Compare TP53-mutant and wild-type tumours across cancer types.

    Retain only cancer types with enough samples in both groups, then
    rank the comparisons by p-value from smallest to largest.
    """
    results = []

    # Analyse each cancer type separately so the TP53 groups are compared
    # within the same tissue context.
    for cancer_type in expr["cancer_type"].unique():
        # Keep tumour samples from the current cancer type.
        tumours = expr[
            (expr["cancer_type"] == cancer_type)
            & (expr["group"] == "tumour")
        ]

        # Count only samples with a known TP53 status and expression value.
        valid_data = tumours[["tp53_status", "log2_tpm"]].dropna()
        n_mut = len(valid_data[valid_data["tp53_status"] == "MUT"])
        n_wt = len(valid_data[valid_data["tp53_status"] == "WT"])

        # Include a cancer type only when both groups meet the chosen minimum.
        if n_mut >= min_group_size and n_wt >= min_group_size:
            # Reuse the same comparison function used in Notebook 02
            # to keep the test and effect-size calculation consistent.
            comparison = compare_groups(
                tumours,
                "tp53_status",
                "MUT",
                "WT",
            )

            # Store the group sizes and comparison results for this cancer type.
            results.append({
                "cancer_type": cancer_type,
                "n_mut": comparison["n_a"],
                "n_wt": comparison["n_b"],
                "difference": comparison["difference"],
                "p_value": comparison["p_value"],
                "significant": comparison["significant"],
            })

    # Keep a consistent set and order of columns, including when results
    # are assembled into the final table.
    columns = [
        "cancer_type",
        "n_mut",
        "n_wt",
        "difference",
        "p_value",
        "significant",
    ]

    across = pd.DataFrame(results, columns=columns)

    # Rank the eligible cancer types by p-value, then reset the row labels
    # so the output has a consecutive index.
    across = across.sort_values("p_value", ascending=True)
    across = across.reset_index(drop=True)

    return across
    raise NotImplementedError("compare_across_cancers")

In [8]:
# CHECK - structure and internal consistency
across = compare_across_cancers(expr, 20)
EXPECTED_ACROSS_COLUMNS = ["cancer_type", "n_mut", "n_wt",
                           "difference", "p_value", "significant"]

run_checks({
    "returns a DataFrame with the required columns":
        lambda: (isinstance(across, pd.DataFrame)
                 and list(across.columns) == EXPECTED_ACROSS_COLUMNS),
    "each cancer type appears once":
        lambda: across["cancer_type"].is_unique,
    "every retained group meets the minimum size":
        lambda: (across[["n_mut", "n_wt"]] >= 20).all().all(),
    "rows are sorted by p-value, smallest first":
        lambda: across["p_value"].is_monotonic_increasing,
    "the index was reset":
        lambda: list(across.index) == list(range(len(across))),
    "significant agrees with p < 0.05 in every row":
        lambda: (across["significant"] == (across["p_value"] < 0.05)).all(),
})

✅ returns a DataFrame with the required columns
✅ each cancer type appears once
✅ every retained group meets the minimum size
✅ rows are sorted by p-value, smallest first
✅ the index was reset
✅ significant agrees with p < 0.05 in every row

6 of 6 checks passed


In [9]:
minimum = 20
eligible_types = []

# Independently identify the cancer types that meet the minimum size
# in both TP53 groups, using samples with known status and expression.
for cancer_type in expr["cancer_type"].unique():
    tumours = expr[
        (expr["cancer_type"] == cancer_type)
        & (expr["group"] == "tumour")
    ]
    valid_data = tumours[["tp53_status", "log2_tpm"]].dropna()

    n_mut = len(valid_data[valid_data["tp53_status"] == "MUT"])
    n_wt = len(valid_data[valid_data["tp53_status"] == "WT"])

    if n_mut >= minimum and n_wt >= minimum:
        eligible_types.append(cancer_type)

reported_types = list(across["cancer_type"])

# Check that the results include every eligible cancer type
# and exclude every type that falls below the threshold.
all_eligible_are_reported = True
for cancer_type in eligible_types:
    if cancer_type not in reported_types:
        all_eligible_are_reported = False

all_reported_are_eligible = True
for cancer_type in reported_types:
    if cancer_type not in eligible_types:
        all_reported_are_eligible = False

assert all_eligible_are_reported and all_reported_are_eligible


# Check the first reported comparison by calculating each TP53 group's
# sample size and median difference directly from the filtered data.
first_cancer = across["cancer_type"][0]
first_tumours = expr[
    (expr["cancer_type"] == first_cancer)
    & (expr["group"] == "tumour")
]

first_valid = first_tumours[["tp53_status", "log2_tpm"]].dropna()

first_mut = first_valid[
    first_valid["tp53_status"] == "MUT"
]["log2_tpm"]

first_wt = first_valid[
    first_valid["tp53_status"] == "WT"
]["log2_tpm"]

direct_difference = first_mut.median() - first_wt.median()

assert across["n_mut"][0] == len(first_mut)
assert across["n_wt"][0] == len(first_wt)
assert abs(across["difference"][0] - direct_difference) < 1e-9

print("The first comparison matches the direct group counts and medians.")

The first comparison matches the direct group counts and medians.


In [ ]:
across

,cancer_type,n_mut,n_wt,difference,p_value,significant
0,BRCA,261,524,0.80260,9.409264e-25,True
1,LIHC,107,249,1.22370,3.453941e-14,True
2,LUAD,257,250,0.98600,2.615002e-13,True
3,PRAD,58,434,0.57425,3.400662e-05,True
4,PAAD,103,67,0.40030,2.528885e-03,True
5,LUSC,396,78,0.35615,3.477483e-03,True
6,GBM,51,98,0.41110,3.584504e-03,True
7,STAD,200,210,0.24420,2.676659e-02,True
8,ESCA,154,26,0.44065,2.993915e-02,True
9,KICH,21,45,0.89410,3.638716e-02,True


---

## Your write-up

Whichever option you chose: what you did, what you found, and — most
importantly — **what it changes about the answer you gave in Notebook
02.**

*I compared CCNB1 expression between TP53-mutant and wild-type tumours in the 12 cancer types with at least 20 samples in each group. Expression was higher in mutant tumours in 10 cancer types, with an unadjusted p < 0.05; LGG and COAD showed small, non-significant differences in the opposite direction. This changes my Notebook 02 conclusion from a finding in LUAD alone to evidence that the increase is also observed across most of the cancer types analysed, although the pattern is not universal, the 12 comparisons increase the chance of false-positive results, and the analysis does not establish causation.*